In [36]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import joblib

from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "raw" / "secom_telemetry_modified.csv"
ARTIFACTS_PATH = ROOT / "artifacts"

torch.manual_seed(42)
np.random.seed(42)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

In [37]:
print("SECTION 1: LOAD DATA")

df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="raise")
df = df.sort_values("timestamp", kind="mergesort").reset_index(drop=True)

if "label" not in df.columns:
    raise ValueError("Dataset must contain a label column.")

if not set(df["label"].dropna().unique()).issubset({-1, 1}):
    raise ValueError("Expected labels -1 for normal and 1 for defects.")

sensor_cols = [c for c in df.columns if c.startswith("sensor_")]

n = len(df)
a = int(n * 0.70)
b = int(n * 0.80)

train_df = df.iloc[:a].copy()
val_df = df.iloc[a:b].copy()
test_df = df.iloc[b:].copy()

print("Total records:", n)
print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Test records:", len(test_df))
print("Training defect prevalence:", (train_df["label"] == 1).mean())
print("Validation defect prevalence:", (val_df["label"] == 1).mean())
print("Test defect prevalence:", (test_df["label"] == 1).mean())

SECTION 1: LOAD DATA


KeyboardInterrupt: 

In [ ]:
print("\nSECTION 2: HANDLE LEGACY PS-L SENSORS")

def get_features(part):
    X = part[sensor_cols].copy()

    # Legacy PS-L records use sensor_0 == 0 as a sentinel.
    legacy = X["sensor_0"].eq(0.0).fillna(False)
    sensor1 = X["sensor_1"].copy()

    X["is_legacy_ps_l"] = legacy.astype(np.float32)

    # Encode legacy sensor_1 codes as categorical indicators.
    for code in (100, 200, 300):
        X[f"legacy_sensor_1_code_{code}"] = (
            legacy & sensor1.eq(float(code))
        ).astype(np.float32)

    # Mask sentinel values, but retain valid sensor_1 measurements
    # for non-legacy records.
    X.loc[legacy, "sensor_0"] = np.nan
    X.loc[legacy, "sensor_1"] = np.nan

    y = (part["label"] == 1).astype(np.int64).to_numpy()
    return X, y


X_train_raw, y_train = get_features(train_df)
X_val_raw, y_val = get_features(val_df)
X_test_raw, y_test = get_features(test_df)

flag_cols = [
    "is_legacy_ps_l",
    "legacy_sensor_1_code_100",
    "legacy_sensor_1_code_200",
    "legacy_sensor_1_code_300"
]

# Keep sensor_1 as a continuous feature for non-legacy records.
numeric_cols = sensor_cols

usable_cols = [
    c for c in numeric_cols
    if X_train_raw[c].notna().any()
    and X_train_raw[c].nunique(dropna=True) > 1
]

if not usable_cols:
    raise ValueError("No usable continuous sensor features were found.")

print("Continuous features:", len(usable_cols))
print("Categorical indicator features:", len(flag_cols))
print("sensor_1 included as a continuous feature:", "sensor_1" in usable_cols)

assert "sensor_1" in usable_cols


SECTION 2: HANDLE LEGACY PS-L SENSORS
Continuous features: 468
Categorical indicator features: 4
sensor_1 included as a continuous feature: True


In [ ]:
print("\nSECTION 3: PREPROCESS DATA")

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

train_imputed = imputer.fit_transform(X_train_raw[usable_cols])
val_imputed = imputer.transform(X_val_raw[usable_cols])
test_imputed = imputer.transform(X_test_raw[usable_cols])

train_scaled = scaler.fit_transform(train_imputed)
val_scaled = scaler.transform(val_imputed)
test_scaled = scaler.transform(test_imputed)


def combine_features(scaled, raw):
    flags = raw[flag_cols].to_numpy(dtype=np.float32)
    return np.column_stack([scaled, flags]).astype(np.float32)


X_train = combine_features(train_scaled, X_train_raw)
X_val = combine_features(val_scaled, X_val_raw)
X_test = combine_features(test_scaled, X_test_raw)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

assert np.isfinite(X_train).all()
assert np.isfinite(X_val).all()
assert np.isfinite(X_test).all()

print("All features are finite.")
print("Imputer and scaler fitted on training data only.")


SECTION 3: PREPROCESS DATA
Training shape: (1096, 472)
Validation shape: (157, 472)
Test shape: (314, 472)
All features are finite.
Imputer and scaler fitted on training data only.


In [ ]:
print("\nSECTION 4: BUILD MODEL")


class WaferAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=32):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )

        self.classifier = nn.Linear(latent_dim, 1)

    def forward(self, x):
        if x.ndim != 2:
            raise ValueError(
                f"Expected a 2D batch, got {tuple(x.shape)}"
            )

        z = self.encoder(x)
        reconstruction = self.decoder(z)
        logits = self.classifier(z).squeeze(-1)

        return reconstruction, logits, z


model = WaferAutoencoder(X_train.shape[1])

sample = torch.tensor(X_train[:8], dtype=torch.float32)
reconstructed, logits, latent = model(sample)

print("Input shape:", tuple(sample.shape))
print("Reconstruction shape:", tuple(reconstructed.shape))
print("Logits shape:", tuple(logits.shape))
print("Latent shape:", tuple(latent.shape))

assert reconstructed.shape == sample.shape
assert logits.shape == (len(sample),)
assert latent.shape[0] == len(sample)


SECTION 4: BUILD MODEL
Input shape: (8, 472)
Reconstruction shape: (8, 472)
Logits shape: (8,)
Latent shape: (8, 32)


In [ ]:
print("\nSECTION 5: TRAIN MODEL WITH EARLY STOPPING")

import copy

train_data = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)

loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True
)

positive_count = max(int(y_train.sum()), 1)
negative_count = len(y_train) - int(y_train.sum())

positive_weight = torch.tensor(
    [negative_count / positive_count],
    dtype=torch.float32
)

bce = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
mse = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

max_epochs = 50
patience = 5
best_val_ap = -1.0
best_epoch = 0
best_state = None
epochs_without_improvement = 0

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)

for epoch in range(max_epochs):
    model.train()
    total_loss = 0.0

    for batch_x, batch_y in loader:
        optimizer.zero_grad()

        reconstructed, batch_logits, _ = model(batch_x)

        assert reconstructed.shape == batch_x.shape
        assert batch_logits.shape == batch_y.shape

        reconstruction_loss = mse(reconstructed, batch_x)
        classification_loss = bce(batch_logits, batch_y)
        loss = reconstruction_loss + classification_loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(batch_x)

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_tensor)[1]
        val_prob = torch.sigmoid(val_logits).cpu().numpy()

    val_ap = average_precision_score(y_val, val_prob)

    print(
        f"Epoch {epoch + 1}/{max_epochs} | "
        f"Train loss: {total_loss / len(train_data):.4f} | "
        f"Validation AP: {val_ap:.4f}"
    )

    if val_ap > best_val_ap + 1e-6:
        best_val_ap = val_ap
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print(f"Early stopping at epoch {epoch + 1}.")
        break

if best_state is None:
    raise RuntimeError("No valid model checkpoint was selected.")

model.load_state_dict(best_state)
model.eval()

print("Best epoch:", best_epoch)
print("Best validation AP:", best_val_ap)
print("Restored best validation checkpoint.")


SECTION 5: TRAIN MODEL WITH EARLY STOPPING
Epoch 1/50 | Train loss: 2.2753 | Validation AP: 0.0456
Epoch 2/50 | Train loss: 2.1522 | Validation AP: 0.0492
Epoch 3/50 | Train loss: 1.8580 | Validation AP: 0.0503
Epoch 4/50 | Train loss: 1.4740 | Validation AP: 0.0518
Epoch 5/50 | Train loss: 1.1943 | Validation AP: 0.0512
Epoch 6/50 | Train loss: 1.0365 | Validation AP: 0.0513
Epoch 7/50 | Train loss: 0.9472 | Validation AP: 0.0526
Epoch 8/50 | Train loss: 1.0020 | Validation AP: 0.0496
Epoch 9/50 | Train loss: 1.0472 | Validation AP: 0.0526
Epoch 10/50 | Train loss: 0.9544 | Validation AP: 0.0498
Epoch 11/50 | Train loss: 0.8718 | Validation AP: 0.0497
Epoch 12/50 | Train loss: 0.8577 | Validation AP: 0.0495
Epoch 13/50 | Train loss: 0.8494 | Validation AP: 0.0500
Epoch 14/50 | Train loss: 0.8409 | Validation AP: 0.0495
Early stopping at epoch 14.
Best epoch: 9
Best validation AP: 0.0526089589422324
Restored best validation checkpoint.


In [ ]:
print("\nSECTION 6: FORWARD-CHAINING EVALUATION AND THRESHOLD POLICY")
print("Fold evaluation uses only the first 80% of the timeline; the final 20% is excluded.")
print("Each fold fits its own imputer, scaler, and model.")

# Expanding-window folds: train on the past, score the next chronological block.
n_fold_data = int(len(df) * 0.80)
fold_specs = [
    (int(n_fold_data * 0.50), int(n_fold_data * 0.6625)),
    (int(n_fold_data * 0.6625), int(n_fold_data * 0.825)),
    (int(n_fold_data * 0.825), n_fold_data),
]
# Expressed as proportions of the full timeline, these are approximately
# train/evaluate: 0-40% / 40-53%, 0-53% / 53-66%, 0-66% / 66-80%.
# The final 20% test split is not used to fit, select, or score folds.
fold_specs = [
    (int(len(df) * 0.40), int(len(df) * 0.53)),
    (int(len(df) * 0.53), int(len(df) * 0.66)),
    (int(len(df) * 0.66), int(len(df) * 0.80)),
]
fold_records = []

for fold_number, (train_end, eval_end) in enumerate(fold_specs, start=1):
    fold_train_df = df.iloc[:train_end].copy()
    fold_eval_df = df.iloc[train_end:eval_end].copy()
    assert len(fold_train_df) > 0 and len(fold_eval_df) > 0
    assert fold_train_df["timestamp"].max() <= fold_eval_df["timestamp"].min()

    Xtr_raw, ytr = get_features(fold_train_df)
    Xev_raw, yev = get_features(fold_eval_df)
    fold_numeric_cols = sensor_cols.copy()
    fold_usable_cols = [
        c for c in fold_numeric_cols
        if Xtr_raw[c].notna().any() and Xtr_raw[c].nunique(dropna=True) > 1
    ]
    if not fold_usable_cols:
        raise ValueError(f"Fold {fold_number} has no usable continuous features.")

    fold_imputer = SimpleImputer(strategy="median")
    fold_scaler = StandardScaler()
    Xtr_imp = fold_imputer.fit_transform(Xtr_raw[fold_usable_cols])
    Xev_imp = fold_imputer.transform(Xev_raw[fold_usable_cols])
    Xtr_scaled = fold_scaler.fit_transform(Xtr_imp)
    Xev_scaled = fold_scaler.transform(Xev_imp)
    Xtr = np.column_stack([Xtr_scaled, Xtr_raw[flag_cols].to_numpy(dtype=np.float32)]).astype(np.float32)
    Xev = np.column_stack([Xev_scaled, Xev_raw[flag_cols].to_numpy(dtype=np.float32)]).astype(np.float32)
    assert np.isfinite(Xtr).all() and np.isfinite(Xev).all()

    torch.manual_seed(420 + fold_number)
    np.random.seed(420 + fold_number)
    fold_model = WaferAutoencoder(Xtr.shape[1])
    fold_dataset = TensorDataset(
        torch.tensor(Xtr, dtype=torch.float32),
        torch.tensor(ytr, dtype=torch.float32)
    )
    fold_loader = DataLoader(
        fold_dataset, batch_size=64, shuffle=True,
        generator=torch.Generator().manual_seed(420 + fold_number)
    )
    pos_count = max(int(ytr.sum()), 1)
    neg_count = len(ytr) - int(ytr.sum())
    fold_bce = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([neg_count / pos_count], dtype=torch.float32)
    )
    fold_mse = nn.MSELoss()
    fold_optimizer = torch.optim.Adam(fold_model.parameters(), lr=0.001)

    for _ in range(20):
        fold_model.train()
        for batch_x, batch_y in fold_loader:
            fold_optimizer.zero_grad()
            reconstructed, fold_logits, _ = fold_model(batch_x)
            fold_loss = fold_mse(reconstructed, batch_x) + fold_bce(fold_logits, batch_y)
            if not torch.isfinite(fold_loss).item():
                raise RuntimeError(f"Non-finite loss in fold {fold_number}.")
            fold_loss.backward()
            fold_optimizer.step()

    fold_model.eval()
    with torch.no_grad():
        fold_scores = torch.sigmoid(
            fold_model(torch.tensor(Xev, dtype=torch.float32))[1]
        ).cpu().numpy()

    fold_records.append({"fold": fold_number, "y": yev, "scores": fold_scores})
    fold_ap = average_precision_score(yev, fold_scores) if len(np.unique(yev)) == 2 else float("nan")
    fold_auc = roc_auc_score(yev, fold_scores) if len(np.unique(yev)) == 2 else float("nan")
    print(
        f"Fold {fold_number}: train={len(ytr)}, eval={len(yev)}, "
        f"defects={int(yev.sum())}, AP={fold_ap:.3f}, AUC={fold_auc:.3f}"
    )

# Pool the chronological out-of-fold predictions. For each fold, flag exactly
# the top k scores at each alert budget, so tied probabilities cannot inflate the budget.
all_oof_y = np.concatenate([r["y"] for r in fold_records])
all_oof_scores = np.concatenate([r["scores"] for r in fold_records])
print("\nPOOLED FORWARD-CHAINING RESULTS (first 80% timeline only)")
print("OOF records:", len(all_oof_y))
print("OOF defects:", int(all_oof_y.sum()))
print("OOF defect prevalence:", float(all_oof_y.mean()))
print("OOF average precision:", average_precision_score(all_oof_y, all_oof_scores))
print("OOF ROC-AUC:", roc_auc_score(all_oof_y, all_oof_scores) if len(np.unique(all_oof_y)) == 2 else "unavailable")
print("Target alert rate | Actual rate | Precision | Recall | TP | FP")
for rate in (0.10, 0.15, 0.20):
    oof_pred = np.zeros(len(all_oof_y), dtype=np.int64)
    offset = 0
    for record in fold_records:
        scores = record["scores"]
        k = max(1, int(np.ceil(len(scores) * rate)))
        top_indices = np.argsort(-scores, kind="mergesort")[:k]
        oof_pred[offset + top_indices] = 1
        offset += len(scores)
    tn, fp, fn, tp = confusion_matrix(all_oof_y, oof_pred, labels=[0, 1]).ravel()
    print(
        f"{rate:>5.0%} | {oof_pred.mean():>11.2%} | "
        f"{precision_score(all_oof_y, oof_pred, zero_division=0):>9.3f} | "
        f"{recall_score(all_oof_y, oof_pred, zero_division=0):>6.3f} | {tp:>2} | {fp:>3}"
    )

# For the currently trained model, derive score cutoffs from validation scores only.
model.eval()
with torch.no_grad():
    val_tensor = torch.tensor(X_val, dtype=torch.float32)
    val_prob = torch.sigmoid(model(val_tensor)[1]).cpu().numpy()

alert_rates = (0.10, 0.15, 0.20)
threshold_by_rate = {}
print("\nVALIDATION THRESHOLDS FOR THE CURRENT MODEL")
print("Target alert rate | Threshold | Actual validation rate | Precision | Recall | TP | FP")
for rate in alert_rates:
    k = max(1, int(np.ceil(len(val_prob) * rate)))
    top_indices = np.argsort(-val_prob, kind="mergesort")[:k]
    candidate = float(val_prob[top_indices[-1]])
    threshold_by_rate[rate] = candidate
    val_pred = (val_prob >= candidate).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(y_val, val_pred, labels=[0, 1]).ravel()
    print(
        f"{rate:>5.0%} | {candidate:.8g} | {val_pred.mean():>21.2%} | "
        f"{precision_score(y_val, val_pred, zero_division=0):>9.3f} | "
        f"{recall_score(y_val, val_pred, zero_division=0):>6.3f} | {tp:>2} | {fp:>2}"
    )

# Primary operating policy: 15% target alert rate; threshold selected on validation only.
threshold = float(threshold_by_rate[0.15])
print("\nPrimary threshold (validation-derived, 15% target alert rate):", threshold)
print("Note: if scores tie at the cutoff, thresholding may flag slightly more than the target rate.")



SECTION 6: FORWARD-CHAINING EVALUATION AND THRESHOLD POLICY
Fold evaluation uses only the first 80% of the timeline; the final 20% is excluded.
Each fold fits its own imputer, scaler, and model.
Fold 1: train=626, eval=204, defects=5, AP=0.078, AUC=0.495
Fold 2: train=830, eval=204, defects=7, AP=0.040, AUC=0.524
Fold 3: train=1034, eval=219, defects=10, AP=0.040, AUC=0.402

POOLED FORWARD-CHAINING RESULTS (first 80% timeline only)
OOF records: 627
OOF defects: 22
OOF defect prevalence: 0.03508771929824561
OOF average precision: 0.04012470501223665
OOF ROC-AUC: 0.40668670172802407
Target alert rate | Actual rate | Precision | Recall | TP | FP
  10% |      10.21% |     0.016 |  0.045 |  1 |  63
  15% |      15.15% |     0.011 |  0.045 |  1 |  94
  20% |      20.10% |     0.024 |  0.136 |  3 | 123

VALIDATION THRESHOLDS FOR THE CURRENT MODEL
Target alert rate | Threshold | Actual validation rate | Precision | Recall | TP | FP
  10% | 0.0038418393 |                10.19% |     0.000 |  0

In [ ]:
print("\nSECTION 7: TEST EVALUATION — ALREADY INSPECTED; EXPLORATORY ONLY")
print("WARNING: Test results are not an untouched final estimate; the test set has already been inspected.")

model.eval()
with torch.no_grad():
    test_tensor = torch.tensor(X_test, dtype=torch.float32)
    test_prob = torch.sigmoid(model(test_tensor)[1]).cpu().numpy()

print("Test records:", len(y_test))
print("Test defects:", int(np.sum(y_test)))
print("Test natural defect prevalence:", float(np.mean(y_test)))
print("Test average precision:", average_precision_score(y_test, test_prob))
print("Test ROC-AUC:", roc_auc_score(y_test, test_prob) if len(np.unique(y_test)) == 2 else "unavailable")
print("\nEXPLORATORY TEST RESULTS AT VALIDATION-DERIVED THRESHOLDS")
print("Target alert rate | Threshold | Flagged | Actual test rate | Precision | Recall | TP | FP")
for rate in alert_rates:
    candidate = threshold_by_rate[rate]
    test_pred_at_rate = (test_prob >= candidate).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(y_test, test_pred_at_rate, labels=[0, 1]).ravel()
    print(
        f"{rate:>5.0%} | {candidate:.8g} | {int(test_pred_at_rate.sum()):>7} | "
        f"{test_pred_at_rate.mean():>15.2%} | "
        f"{precision_score(y_test, test_pred_at_rate, zero_division=0):>9.3f} | "
        f"{recall_score(y_test, test_pred_at_rate, zero_division=0):>6.3f} | {tp:>2} | {fp:>3}"
    )

test_pred = (test_prob >= threshold).astype(np.int64)
print("\nPrimary 15%-target policy (exploratory; test already inspected)")
print("Flagged:", int(test_pred.sum()), "/", len(test_pred))
print("Actual test alert rate:", float(test_pred.mean()))
print("Precision:", precision_score(y_test, test_pred, zero_division=0))
print("Recall:", recall_score(y_test, test_pred, zero_division=0))
print("F1:", f1_score(y_test, test_pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))



SECTION 7: TEST EVALUATION — ALREADY INSPECTED; EXPLORATORY ONLY
Test records: 314
Test defects: 17
Test natural defect prevalence: 0.054140127388535034
Test average precision: 0.09348036081939015
Test ROC-AUC: 0.6599326599326599

EXPLORATORY TEST RESULTS AT VALIDATION-DERIVED THRESHOLDS
Target alert rate | Threshold | Flagged | Actual test rate | Precision | Recall | TP | FP
  10% | 0.0038418393 |      10 |           3.18% |     0.000 |  0.000 |  0 |  10
  15% | 0.0010033317 |      19 |           6.05% |     0.105 |  0.118 |  2 |  17
  20% | 0.00053528388 |      26 |           8.28% |     0.115 |  0.176 |  3 |  23

Primary 15%-target policy (exploratory; test already inspected)
Flagged: 19 / 314
Actual test alert rate: 0.06050955414012739
Precision: 0.10526315789473684
Recall: 0.11764705882352941
F1: 0.1111111111111111
Confusion matrix:
[[280  17]
 [ 15   2]]


In [ ]:
print("\nSECTION 8: SAVE MODEL AND PREPROCESSOR")

ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "input_dim": X_train.shape[1],
        "latent_dim": 32,
        "threshold": threshold,
        "usable_cols": usable_cols,
        "flag_cols": flag_cols
    },
    ARTIFACTS_PATH / "wafer_model.pt"
)

joblib.dump(
    {
        "imputer": imputer,
        "scaler": scaler,
        "usable_cols": usable_cols,
        "flag_cols": flag_cols
    },
    ARTIFACTS_PATH / "preprocessor.joblib"
)

print("Model saved:", ARTIFACTS_PATH / "wafer_model.pt")
print("Preprocessor saved:", ARTIFACTS_PATH / "preprocessor.joblib")


SECTION 8: SAVE MODEL AND PREPROCESSOR
Model saved: a:\05_Wafer\wafer-guard\artifacts\wafer_model.pt
Preprocessor saved: a:\05_Wafer\wafer-guard\artifacts\preprocessor.joblib


In [ ]:
print("\nSECTION 9: AUDIT SUMMARY")
print("Evaluation split: chronological 70% / 10% / 20%")
print("Forward-chaining fold evaluation: expanding windows within first 80% of timeline")
print("Final 20% test set excluded from fold training and threshold selection: Yes")
print("Test set has already been inspected; test metrics are exploratory only: Yes")
print("Test set balanced artificially: No")
print("Preprocessing fitted on training data only: Yes")
print("Legacy sensor_1 codes used as continuous values: No")
print("Model deployment recommendation: Do not deploy without acceptable defect recall and stable forward-chaining results.")



SECTION 9: AUDIT SUMMARY
Evaluation split: chronological 70% / 10% / 20%
Forward-chaining fold evaluation: expanding windows within first 80% of timeline
Final 20% test set excluded from fold training and threshold selection: Yes
Test set has already been inspected; test metrics are exploratory only: Yes
Test set balanced artificially: No
Preprocessing fitted on training data only: Yes
Legacy sensor_1 codes used as continuous values: No
Model deployment recommendation: Do not deploy without acceptable defect recall and stable forward-chaining results.


In [ ]:

import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
)

print("GRADIENT BOOSTING: FORWARD-CHAINING TUNING")

# Keep the final 20% test set untouched.
fold_specs_gb = [
    (int(len(df) * 0.40), int(len(df) * 0.53)),
    (int(len(df) * 0.53), int(len(df) * 0.66)),
    (int(len(df) * 0.66), int(len(df) * 0.80)),
]

configs = [
    {
        "max_iter": 100,
        "learning_rate": 0.03,
        "max_leaf_nodes": 7,
        "l2_regularization": 1.0,
    },
    {
        "max_iter": 150,
        "learning_rate": 0.03,
        "max_leaf_nodes": 15,
        "l2_regularization": 5.0,
    },
    {
        "max_iter": 100,
        "learning_rate": 0.05,
        "max_leaf_nodes": 7,
        "l2_regularization": 5.0,
    },
    {
        "max_iter": 150,
        "learning_rate": 0.05,
        "max_leaf_nodes": 15,
        "l2_regularization": 1.0,
    },
]

tuning_results = []

for ci, cfg in enumerate(configs, start=1):
    print(f"\n{'=' * 55}")
    print(f"CONFIGURATION {ci}: {cfg}")

    fold_results = []
    all_y = []
    all_scores = []

    for fold, (train_end, eval_end) in enumerate(
        fold_specs_gb, start=1
    ):
        train = df.iloc[:train_end].copy()
        valid = df.iloc[train_end:eval_end].copy()

        Xtr_raw, ytr = get_features(train)
        Xev_raw, yev = get_features(valid)

        # Preserve sensor_1 for non-legacy records.
        # Legacy sensor codes are handled by get_features().
        cols = [
            c for c in sensor_cols
            if Xtr_raw[c].notna().any()
            and Xtr_raw[c].nunique(dropna=True) > 1
        ]

        # Fit preprocessing only on this fold's training data.
        imputer = SimpleImputer(strategy="median")
        Xtr_num = imputer.fit_transform(Xtr_raw[cols])
        Xev_num = imputer.transform(Xev_raw[cols])

        Xtr = np.column_stack([
            Xtr_num,
            Xtr_raw[flag_cols].to_numpy(dtype=np.float32),
        ])
        Xev = np.column_stack([
            Xev_num,
            Xev_raw[flag_cols].to_numpy(dtype=np.float32),
        ])

        model = HistGradientBoostingClassifier(
            **cfg,
            class_weight="balanced",
            random_state=42,
        )
        model.fit(Xtr, ytr)

        scores = model.predict_proba(Xev)[:, 1]

        ap = average_precision_score(yev, scores)
        auc = (
            roc_auc_score(yev, scores)
            if len(np.unique(yev)) == 2
            else float("nan")
        )

        print(
            f"Fold {fold}: train={len(ytr)}, "
            f"eval={len(yev)}, defects={int(yev.sum())}, "
            f"AP={ap:.4f}, AUC={auc:.4f}"
        )

        fold_results.append((ap, auc))
        all_y.append(yev)
        all_scores.append(scores)

    # Combine only the forward-chaining validation folds.
    y = np.concatenate(all_y)
    scores = np.concatenate(all_scores)

    pooled_ap = average_precision_score(y, scores)
    pooled_auc = (
        roc_auc_score(y, scores)
        if len(np.unique(y)) == 2
        else float("nan")
    )

    tuning_results.append({
        "config": cfg,
        "ap": pooled_ap,
        "auc": pooled_auc,
        "fold_results": fold_results,
    })

    print("\nPOOLED VALIDATION RESULTS")
    print("Records:", len(y))
    print("Defects:", int(y.sum()))
    print(f"Average Precision: {pooled_ap:.4f}")
    print(f"ROC-AUC: {pooled_auc:.4f}")

    print("\nAlert rate | Precision | Recall | TP | Flagged")

    for rate in (0.10, 0.15, 0.20):
        k = max(1, int(np.ceil(len(scores) * rate)))
        selected = np.argsort(-scores, kind="mergesort")[:k]

        pred = np.zeros(len(y), dtype=int)
        pred[selected] = 1

        tp = int(np.sum((pred == 1) & (y == 1)))

        print(
            f"{rate:>9.0%} | "
            f"{precision_score(y, pred, zero_division=0):.3f} | "
            f"{recall_score(y, pred, zero_division=0):.3f} | "
            f"{tp} | {int(pred.sum())}"
        )

# Rank configurations by pooled Average Precision.
tuning_results.sort(key=lambda r: r["ap"], reverse=True)

print("\n" + "=" * 55)
print("CONFIGURATION COMPARISON")
print("Rank | AP     | ROC-AUC | Parameters")

for rank, result in enumerate(tuning_results, start=1):
    print(
        f"{rank:>4} | "
        f"{result['ap']:.4f} | "
        f"{result['auc']:.4f} | "
        f"{result['config']}"
    )

best = tuning_results[0]

print("\nBEST CONFIGURATION BY POOLED AVERAGE PRECISION")
print("Average Precision:", round(best["ap"], 4))
print("ROC-AUC:", round(best["auc"], 4))
print("Parameters:", best["config"])

print("\nFold AP values for the winning configuration:")
print([round(ap, 4) for ap, auc in best["fold_results"]])

print(
    "\nNote: These are forward-chaining validation results, "
    "not final test-set results."
)



GRADIENT BOOSTING: FORWARD-CHAINING EVALUATION
Fold 1: train=626, eval=204, defects=5, AP=0.207, AUC=0.725
Fold 2: train=830, eval=204, defects=7, AP=0.297, AUC=0.647
Fold 3: train=1034, eval=219, defects=10, AP=0.054, AUC=0.533

POOLED GRADIENT BOOSTING RESULTS
Records: 627
Defects: 22
Average precision: 0.12483928827719168
ROC-AUC: 0.5901577761081893

Alert rate | Precision | Recall | TP | Flagged
      10% | 0.095 | 0.273 | 6 | 63

SCORE DIAGNOSTICS
      15% | 0.063 | 0.273 | 6 | 95

SCORE DIAGNOSTICS
      20% | 0.048 | 0.273 | 6 | 126

SCORE DIAGNOSTICS
Unique prediction scores: 627
Total scores: 627
Largest score frequency: 1
Defect score range: (0.011742140475985766, 0.4191762760465612)
Normal score range: (0.006077148381496622, 0.4045280369405986)

QUICK GRADIENT BOOSTING TUNING
Config 1: AP=0.1441, AUC=0.6437, params={'max_iter': 100, 'learning_rate': 0.03, 'max_leaf_nodes': 7, 'l2_regularization': 1.0}
Config 2: AP=0.1324, AUC=0.6342, params={'max_iter': 150, 'learning_rate